In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:13pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:5xpt;} 
</style>
"""))

**<font size="6" color="orange">ch2_Ollama_LLM활용의 기본 개념(LangChaing)</font>**

# 1. LLM을 활용한 답변 생성
## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT(open ai API), Claude같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용
### ollama.com 설치 -> 모델 pull
- cmd창에서 ollama pull deepseek-r1:1.5b

In [3]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of France?')
result # AIMessage
# content : 실제 답변
# response_metadata : 모델 실행에 대한 상세 정보 (전체 소요시간, 모델 로딩 시간, 처리토큰수)

AIMessage(content='\n\nThe capital of France remains Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-09-10T07:56:05.3578532Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14315597200, 'load_duration': 9769222200, 'prompt_eval_count': 10, 'prompt_eval_duration': 99757000, 'eval_count': 288, 'eval_duration': 4431233000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a08a50-df7b-77b2-83dc-452c080a538b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 288, 'total_tokens': 298})

In [5]:
print(result.content)



The capital of France remains Paris.


### 모델 pull
- ollama pull llama3.2:1b
- ollama 모델은 공식적으로 한글 지원을 하지 않음.(ollama run llama3.1:405b : 한글지원 기능 → ollama run llama3.2:3b : 한글지원 일부 가능)
- exaone 모델은 공식적으로 한글지원 가능.(ollama pull exaone3.5:2.4b

- 모델 저장 경로 : C:\Users\water(내켬퓨터이름)\.ollama

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b',
                 temperature=0.2,
                 top_k=40,
                 top_p=0.9,
                 # num_ctx=30 # 컨텍스트 윈도우(입력 토근과 출력 토근)
                )
result = llm.invoke('what is the capital of Korea?')
result

AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T00:44:44.1816246Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9589294300, 'load_duration': 9379015500, 'prompt_eval_count': 32, 'prompt_eval_duration': 68729000, 'eval_count': 8, 'eval_duration': 121532000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08dec-6394-7c43-8e3b-fba1b449557d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [3]:
result.content

'The capital of Korea is Seoul.'

In [6]:
llm = ChatOllama(model='exaone3.5:2.4b')
result = llm.invoke('한국 수도는 어디예요?')
result.content

'한국의 수도는 **서울**입니다. 서울은 정치, 경제, 문화의 중심지로 국가 행정의 핵심 역할을 담당하고 있습니다.'

## 2) open ai 모델 활용
- pip install langchain-openai

In [9]:
# 환경변수(`OPENAI_API_KEY` or `OPENAI_ADMIN_KEY`)
from dotenv import load_dotenv
import os
load_dotenv()
# print(os.getenv('OPENAI_API_KEY'))
# print(os.environ['OPENAI_API_KEY'])

True

In [11]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano",
                 # api_key=os.getenv('OPENAI_API_KEY')
                )
result = llm.invoke('What is the capital of Korea?')
# result = llm.invoke('한국의 수도가 어디야?')
result.content

'The capital of South Korea is Seoul.'

In [ ]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
# ANTHROPIC_API_KEY environment variable
# llm.invoke('What is the capital of Korea?')

# 2. Langchaing style로 Prompt 작성하기
- Prompt : llm호출시 쓰는 질문
## 1) 기본 prompt template 사용
- PromptTemplate을 사용하는 변수가 포함된 templat을 작성하면 PromptValue를 만들 수 있다.

In [3]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0)
llm.invoke("What is the capital of Korea?")
# 프롬프트 가능한 타입 : str, PromptValue, list of BaseMessages

AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:02:25.7880439Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10250171200, 'load_duration': 10044265500, 'prompt_eval_count': 32, 'prompt_eval_duration': 74476000, 'eval_count': 8, 'eval_duration': 123062000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e6a-70ea-7631-b643-6057af71026b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [4]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(template="What is the capital of {country}?", # {}안을 새로운 값으로 대체 가능함.
                                 input_variables = ['country'],
                                )
prompt = prompt_template.invoke({"country":"Korea"})
print(1,prompt)
prompt = prompt_template.invoke("Korea")
print(2, prompt)
llm.invoke(prompt)

1 text='What is the capital of Korea?'
2 text='What is the capital of Korea?'


AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:02:26.0121836Z', 'done': True, 'done_reason': 'stop', 'total_duration': 170007000, 'load_duration': 6830600, 'prompt_eval_count': 32, 'prompt_eval_duration': 17919000, 'eval_count': 9, 'eval_duration': 135338000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e6a-992f-7b81-9ee9-43ac4fd833a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41})

In [15]:
country = input('수도를 알고 싶은 나라는(영어)?')
llm.invoke(prompt_template.invoke(country))

수도를 알고 싶은 나라는(영어)?홍콩


AIMessage(content='The capital of Hong Kong is Victoria, but the city is officially known as Hong Kong, Hong Kong.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T02:35:58.7598927Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3201083500, 'load_duration': 6253500, 'prompt_eval_count': 33, 'prompt_eval_duration': 2817449000, 'eval_count': 22, 'eval_duration': 369193000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e52-5523-7c51-ac67-d7be1529ba6c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 22, 'total_tokens': 55})

In [1]:
def answer(country):
    '나라명을 입력받아 llm에 수도명을 받아 return'
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm = ChatOllama(model='llama3.2:1b')
    prompt_template = PromptTemplate(template='What is the capital of {country}?',
                                     imput_variables = ['country'] 
                                    )
    result = llm.invoke(prompt_template.invoke(country))
    return result.content

In [2]:
country = input("수도를 알고 싶은 나라는 (영어)?")
answer(country)

수도를 알고 싶은 나라는 (영어)?일본


'The capital of Japan is Tokyo.'

## 2) 메세지 기반 프롬프트 작성
- list of BaseMessages
- BaseMessages 상속받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage
- [BaseMessage객체, BaseMessage객체, BaseMessage객체, ...]


In [10]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant!"), # llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Paris.'),
    HumanMessage(content="What is the capital of Korea?") # 질문 답변 예제(few shot)
]
llm.invoke(message_list)

AIMessage(content='The capital of South Korea is Seoul, and the capital of North Korea is Pyongyang.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:19:51.8319495Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11586079700, 'load_duration': 9535745700, 'prompt_eval_count': 86, 'prompt_eval_duration': 1743702000, 'eval_count': 18, 'eval_duration': 297997000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e7a-61d3-7033-9b52-683f7c9a2a95-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 18, 'total_tokens': 104})

## 3) ChatPromptTemplate 사용 👍 확장성 용이 👍

In [14]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    SystemMessage(content="You are a helpful assistant!"), # llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Paris.'),
    HumanMessage(content="What is the capital of {country}?") # 질문 답변 예제(few shot)
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)

프롬프트 : messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of {country}?', additional_kwargs={}, response_metadata={})]


In [15]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    ("system","You are a helpful assistant!"),
    ("human","What is the capital of Italy?"),
    ("ai","The capital of Italy is Rome."),
    ("human","What is the capital of France?"),
    ("ai","The capital of Italy is Paris."),
    ("human","What is the capital of {country}?"),
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)


프롬프트 : messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [16]:
llm.invoke(prompt)
# llm.invoke(chatPromptTemplate.invoke({'country':'Korea'}))
# llm.invoke(chatPromptTemplate.invoke({'Korea'}))

AIMessage(content='The capital of South Korea is Seoul. The capital of North Korea is Pyongyang.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:51:12.3823336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12552326000, 'load_duration': 12131337400, 'prompt_eval_count': 86, 'prompt_eval_duration': 122709000, 'eval_count': 17, 'eval_duration': 286945000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e97-0ff1-7452-9751-0e1b23333a79-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 17, 'total_tokens': 103})

# 3. 답변 형식 컨트롤하기
- invoke 실행 결과 AIMessage() → string, json 변환해주는 OutputParser 이용

## 1) 문자열 출력 Parser 이용
- StrOutputParser를 이용하여 LLM출력(AIMessage)를 단순 문자열로 변환

In [20]:
# from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(template='What is the capital of {country}? Return the name of the city only.',
                                 input_variables=['country']
                                 
                                )
# 프롬프트 템플릿에 값을 주입
prompt = prompt_template.invoke({'country':'Korea'})
print('프롬프트 :', prompt)
# llm에 질문
aimessage = llm.invoke(prompt)
# aimessage 중 답변만 문자로 받기
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print('parser 결과 : ', result)

프롬프트 : text='What is the capital of Korea? Return the name of the city only.'
parser 결과 :  Seoul


In [22]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))
output_parser.invoke(llm.invoke(prompt_template.invoke('Korea')))

'Seoul'

In [24]:
chatPromptTemplate = ChatPromptTemplate([
    ("system","You are a helpful assistant!"),
    ("human","What is the capital of Italy?"),
    ("ai","The capital of Italy is Rome."),
    ("human","What is the capital of France?"),
    ("ai","The capital of Italy is Paris."),
    ("human","What is the capital of {country} Return the name of the city only.?")
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})))

'Seoul'

## 2) Json Output Parser 이용
- {'name' : '홍', 'age':20}

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

llm = ChatOllama(model='deepseek-r1:1.5b')
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
            - Capital
            - Population
            - Language
            - Currency
        Return ONLY a valid JSON object with no additional text.
        Example format:
        {{"Capital":"Seoul", "Population":"50 million", "Language":"Korea", "Currency":"won"}}""",
    input_variables = ['country']
)
prompt = country_detail_prompt.invoke({'country':'France'})
aimessage = llm.invoke(prompt)
output_parsers = JsonOutputParser()
result = output_parsers.invoke(aimessage)
print(type(result), result)

<class 'dict'> {'Capital': 'Paris', 'Population': '67 million', 'Language': 'French', 'Currency': 'euro'}


In [8]:
info = output_parsers.invoke(llm.invoke(country_detail_prompt.invoke('France')))
info

{'Capital': 'Paris',
 'Population': '67 million',
 'Language': 'French',
 'Currency': 'Euro'}

## 3) 구조화된 객체로 반환하기
- Pydantic 모델(pip install pydantic(설치) → pip show pydantic(확인))을 사용하며 llm출력을 구조화된 형식으로 받기(JsonParser 좀 안정적임)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리


In [10]:
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
user = User('1', "고길동")
print(user)
print(user.id, user.name, user.is_active)

1 고길동 True


In [13]:
from pydantic import BaseModel, Field
class User(BaseModel) :
    # gt=0 : id>0 / lt=0 : id<0 / ge=0 : id>=0 / le=0 : id<=0
    id:int         = Field(gt=0, description='id')
    name:str       = Field(min_length=2,description='이름')
    is_active:bool = Field(default=True, description='id활성화')
user = User(id=1, name='고길동', is_active=True)
print(user)

id=1 name='고길동' is_active=True


In [24]:
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
            - Capital
            - Population
            - Language
            - Currency
        Return ONLY a valid JSON object with no additional text.
        Example format:
        {{"Capital":"Seoul", "Population":"50 million", "Language":"Korea", "Currency":"won"}}""",
    input_variables = ['country']
)
class CountryDetail(BaseModel):
    capital:str = Field(description='the capital of the country')
    population:int = Field(description='the population of the country')
    language:str = Field(description='the language of the country')
    currency:str = Field(description='the currency of the country')
# 출력파서 + llm
structedallm = llm.with_structured_output(CountryDetail)
info = structedallm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(type(info))
print(info)
print(info.capital, info.population, info.language, info.currency)
print(info.model_dump()) # 객체를 dict로

<class '__main__.CountryDetail'>
capital='Seoul' population=138000000 language='Korean' currency='won'
Seoul 138000000 Korean won
{'capital': 'Seoul', 'population': 138000000, 'language': 'Korean', 'currency': 'won'}


In [38]:
aimessage = llm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(aimessage.content)



{"Capital":"Seoul", "Population":"130 million", "Language":"Korea", "Currency":"won"}


# 4. LCEL(LangChain Expression Language)을 활용한 렝체인 생성
## 1) 문자열 출력 파서 사용


In [36]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 명시적인 지시사항이 포함된 프롬프르
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables =['country']
)
output_parsers = StrOutputParser()
output_parsers.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

'\n\nThe capital of Korea is Gyeongbokgao.'

## 2) LCEL을 사용한 체인 구성
- ( | ) 사용

In [30]:
# 프롬프트 템플렛 → llm → 출력파서를 연결 시키는 체인 생성
capital_chain = prompt_template | llm | output_parsers
# 생성된 체인 invoke
capital_chain.invoke({'country':'Korea'})

'\n\nThe capital of Korea is Gwanggole, located in Jeollanamgae, which is the northern part of South Korea.'

In [31]:
type(capital_chain)

langchain_core.runnables.base.RunnableSequence

## 3) 복합 체인 구성
- 여러 단계의 추론이 필요한 경우 (체인 연결)

In [32]:
# 나라에 대한 설명 → 나라이름 출력
country_prompt = PromptTemplate(template="Guess the name of the country based on the follwinf information:{information} Return the name of the country only",
                                input_variables=['information']
                               )
output_parsers.invoke(llm.invoke(country_prompt.invoke({'information':"This country is very famous for its wine"})))

'\n\nThe country known for its famous wine is France.'

In [34]:
# 나라명 추측 체인
country_chain = country_prompt | llm | outputParser
country_chain.invoke({'information':'This country is very famous for its wine'})

'\n\nFrance'

In [39]:
# 국가 설명 → 국가명 → 그 국가 수도명
final_chain = country_chain | capital_chain
final_chain.invoke({'information':'This country is very famous for its wine'})

'\n\nThe capital of the country famous for its wine is France, which has Paris as its capital.\n\n**Answer:** Paris'

In [40]:
# 국가 설명 → 국가명 → 그 국가 수도명
final_chain = {'country':country_chain} | capital_chain
final_chain.invoke({'information':'This country is very famous for its wine'})

'\n\nThe capital of France is Paris.'

```
LLM호출: ChatOllama(llama3.2:1b / exaone3.5:2.4b), ChatOpenAI
LLM호출에 필요한 프롬프트 템플릿 : PromptTemplate, ChatPromptTemplate(few, shot, 페르소나 설정)
LLM결과를 변환 outputParser : str/JSON/pydentic 클래스 생성하여 LLM의 Structed outputParser 이용
위 모두 runnable로부터 상속받는 invoke 가능 → LangChain으로 연결 가능(→RAG)
```

# 5. 생성형 AI 평가
- 나라이름 → 그 나라에서 가장 유명한 음식(출력) → 그 음식의 레시피(출력)
1. 체인 = 음식 출력
2. 체인 = 음식의 레시피 출력

- 최종 체인
나라이름 → 그 나라에 가장 유명한 음식의 레시피